In [1]:
import pandas as pd
from datetime import datetime, timedelta
import requests
import os
from zoneinfo import ZoneInfo
import time

In [3]:
def get_flight_data(icao_list):
    # It is highly recommended to use an environment variable instead of hardcoding your key!
    api_key = os.getenv("RAPIDAPI_KEY") 

    berlin_timezone = ZoneInfo('Europe/Berlin')
    today = datetime.now(berlin_timezone).date()
    tomorrow = (today + timedelta(days=1))

    flight_items = []

    for icao in icao_list:
        times = [["00:00", "11:59"],
                 ["12:00", "23:59"]]

        for time_slot in times:
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time_slot[0]}/{tomorrow}T{time_slot[1]}"

            querystring = {
                "withLeg": "true",
                "direction": "Arrival",
                "withCancelled": "false",
                "withCodeshared": "true",
                "withCargo": "false",
                "withPrivate": "false"
            }

            headers = {
                'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
                'x-rapidapi-key': api_key
            }

            response = requests.get(url, headers=headers, params=querystring)
            
            # 1. Check if the response was successful
            if response.status_code != 200:
                print(f"Error {response.status_code} for {icao} during {time_slot[0]}-{time_slot[1]}: {response.text}")
                time.sleep(1.5) # Still sleep to avoid hammering the API
                continue

            flights_json = response.json()
            retrieval_time = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")

            # 2. Use .get() with a fallback empty list to prevent KeyError
            arrivals = flights_json.get("arrivals", [])

            for item in arrivals:
                # Safely extract departure and arrival nested data structures
                departure = item.get("departure", {})
                departure_airport = departure.get("airport", {}) if departure else {}
                arrival = item.get("arrival", {})

                flight_item = {
                    "arrival_airport_icao": icao,
                    "departure_airport_icao": departure_airport.get("icao", None),
                    "departure_airport_name": departure_airport.get("name", None),
                    "scheduled_arrival_time": arrival.get("scheduledTime", {}).get("local", None) if arrival else None,
                    "flight_number": item.get("number", None),
                    "data_retrieved_at": retrieval_time
                }
                flight_items.append(flight_item)

            # 3. Pause briefly to respect the BASIC plan per-second rate limits
            time.sleep(1.5)

    # If no data was collected at all, return an empty dataframe
    if not flight_items:
        print("No flight schedules found for the requested criteria.")
        return pd.DataFrame()

    flights_df = pd.DataFrame(flight_items)
    
    # Safely clean strings and convert to datetime
    flights_df["scheduled_arrival_time"] = flights_df["scheduled_arrival_time"].str[:-6]
    flights_df["scheduled_arrival_time"] = pd.to_datetime(flights_df["scheduled_arrival_time"])
    flights_df["data_retrieved_at"] = pd.to_datetime(flights_df["data_retrieved_at"])

    return flights_df

In [4]:
icao_list = ["EDDB"]

flights_to_db = get_flight_data(icao_list)
flights_to_db

,arrival_airport_icao,departure_airport_icao,departure_airport_name,scheduled_arrival_time,flight_number,data_retrieved_at
0,EDDB,OLBA,Beirut,2026-05-29 06:25:00,SR 149,2026-05-28 15:07:07
1,EDDB,LTAJ,Gaziantep,2026-05-29 06:45:00,XQ 1766,2026-05-28 15:07:07
2,EDDB,ZBAA,Beijing,2026-05-29 06:45:00,HU 489,2026-05-28 15:07:07
3,EDDB,LROP,Bucharest,2026-05-29 07:05:00,W4 3109,2026-05-28 15:07:07
4,EDDB,LZIB,Bratislava,2026-05-29 07:15:00,W6 7037,2026-05-28 15:07:07
...,...,...,...,...,...,...
503,EDDB,EIDW,Dublin,2026-05-29 23:00:00,FR 5418,2026-05-28 15:07:09
504,EDDB,LGKO,Kos Island,2026-05-29 23:05:00,A3 3966,2026-05-28 15:07:09
505,EDDB,LGKO,Kos Island,2026-05-29 23:05:00,EW 8671,2026-05-28 15:07:09
506,EDDB,LBBG,Burgas,2026-05-29 23:05:00,SR 7359,2026-05-28 15:07:09


In [5]:
flights_to_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   arrival_airport_icao    508 non-null    str           
 1   departure_airport_icao  507 non-null    str           
 2   departure_airport_name  508 non-null    str           
 3   scheduled_arrival_time  508 non-null    datetime64[us]
 4   flight_number           508 non-null    str           
 5   data_retrieved_at       508 non-null    datetime64[us]
dtypes: datetime64[us](2), str(4)
memory usage: 23.9 KB


## Create "flight" table in the database

```sql
CREATE TABLE flights(
	flight_id INT AUTO_INCREMENT,
    arrival_airport_icao VARCHAR(10),
    departure_airport_icao VARCHAR(10),
    departure_airport_name VARCHAR(30),
    scheduled_arrival_time DATETIME,
    flight_number VARCHAR(30),
    data_retrieved_at DATETIME,
    PRIMARY KEY (flight_id),
    FOREIGN KEY (arrival_airport_icao) REFERENCES airports(icao)
);


Push the 'flights_to_db' to the flights table in the SQL database 


In [6]:
schema = "wikipedia"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [7]:
flights_to_db.to_sql('flights',
                    if_exists='append',
                    con=connection_string,
                    index=False)

508

In [8]:
pd.read_sql("flights", con=connection_string)

,flight_id,arrival_airport_icao,departure_airport_icao,departure_airport_name,scheduled_arrival_time,flight_number,data_retrieved_at
0,1,EDDB,OLBA,Beirut,2026-05-29 06:25:00,SR 149,2026-05-28 15:07:07
1,2,EDDB,LTAJ,Gaziantep,2026-05-29 06:45:00,XQ 1766,2026-05-28 15:07:07
2,3,EDDB,ZBAA,Beijing,2026-05-29 06:45:00,HU 489,2026-05-28 15:07:07
3,4,EDDB,LROP,Bucharest,2026-05-29 07:05:00,W4 3109,2026-05-28 15:07:07
4,5,EDDB,LZIB,Bratislava,2026-05-29 07:15:00,W6 7037,2026-05-28 15:07:07
...,...,...,...,...,...,...,...
503,504,EDDB,EIDW,Dublin,2026-05-29 23:00:00,FR 5418,2026-05-28 15:07:09
504,505,EDDB,LGKO,Kos Island,2026-05-29 23:05:00,A3 3966,2026-05-28 15:07:09
505,506,EDDB,LGKO,Kos Island,2026-05-29 23:05:00,EW 8671,2026-05-28 15:07:09
506,507,EDDB,LBBG,Burgas,2026-05-29 23:05:00,SR 7359,2026-05-28 15:07:09
